## Script to convert a VIA dataset (like Greenland OOD dataset) to a JSON file compatible with COCO format

In [ ]:
import pandas as pd
import json
import os
import cv2
import numpy as np
from tqdm import tqdm
from pycocotools import mask as mask_util

DATASETS_PATH = os.environ.get("ZERO_SHOT_DATA_DIR", "./data")
CSV_FILE = os.path.join(DATASETS_PATH, "greenland_test_data/via_region_data (58).csv")
IMAGE_DIR = os.path.join(DATASETS_PATH, "greenland_test_data", "Annotated files")
OUTPUT_JSON = os.path.join(DATASETS_PATH, "greenland_test_data/coco_region_data.json")
CATEGORY_NAME = "insect"
CATEGORY_ID = 1

def create_coco(csv_path, img_dir, category_name, category_id):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: CSV file '{csv_path}' not found.")
        return

    coco_output = {
        "info": {
            "description": "Converted from VIA CSV (RLE Format)",
            "year": 2026,
            "date_created": "2026-02-12"
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": [
            {"id": category_id, "name": category_name, "supercategory": "object"}
        ]
    }

    filename_to_id = {}
    current_img_id = 1
    current_ann_id = 1
    skipped_rows = 0

    print(f"Processing {len(df)} rows from CSV...")

    for _, row in tqdm(df.iterrows(), total=df.shape[0]):
        filename = row['filename']
        
        if filename not in filename_to_id:
            img_path = os.path.join(img_dir, filename)
            
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                if img is None:
                    skipped_rows += 1
                    continue
                height, width = img.shape[:2]
            else:
                skipped_rows += 1
                continue

            image_info = {
                "id": current_img_id,
                "file_name": filename,
                "width": width,
                "height": height
            }
            coco_output["images"].append(image_info)
            filename_to_id[filename] = current_img_id
            current_img_id += 1
        
        image_id = filename_to_id[filename]
        img_info = next(img for img in coco_output['images'] if img['id'] == image_id)
        height, width = img_info['height'], img_info['width']

        try:
            region_data = json.loads(row['region_shape_attributes'])
        except (json.JSONDecodeError, TypeError):
            continue

        if not region_data or 'all_points_x' not in region_data:
            continue

        points_x = region_data['all_points_x']
        points_y = region_data['all_points_y']

        mask = np.zeros((height, width), dtype=np.uint8)
        
        poly_points = np.array(list(zip(points_x, points_y)), dtype=np.int32)
        cv2.fillPoly(mask, [poly_points], 1)
        
        rle = mask_util.encode(np.asfortranarray(mask))
        
        rle['counts'] = rle['counts'].decode('utf-8')

        area = float(mask_util.area(rle))
        bbox = mask_util.toBbox(rle).tolist()

        annotation = {
            "id": current_ann_id,
            "image_id": image_id,
            "category_id": category_id,
            "segmentation": rle,
            "area": area,
            "bbox": bbox,
            "iscrowd": 0
        }

        coco_output["annotations"].append(annotation)
        current_ann_id += 1

    with open(OUTPUT_JSON, 'w') as f:
        json.dump(coco_output, f, indent=4)
    
    print(f"Images: {len(coco_output['images'])}")
    print(f"Annotations: {len(coco_output['annotations'])}")
    print(f"Skipped rows: {skipped_rows}")
    print(f"Saved to: {OUTPUT_JSON}")

create_coco(
    csv_path=CSV_FILE, 
    img_dir=IMAGE_DIR, 
    category_name=CATEGORY_NAME, 
    category_id=CATEGORY_ID
)

In [ ]:
import json
import os
import cv2
import numpy as np
from matplotlib import pyplot as plt
from pycocotools import mask as mask_util
from collections import defaultdict

def plot_annotations_with_crops(coco_json, img_dir, num_images=5, max_crops_per_img=10):
    with open(coco_json) as f:
        coco_data = json.load(f)

    annotated_img_ids = {ann['image_id'] for ann in coco_data['annotations']}
    images = [img for img in coco_data['images'] if img['id'] in annotated_img_ids][:num_images]

    img_to_anns = defaultdict(list)
    for ann in coco_data['annotations']:
        img_to_anns[ann['image_id']].append(ann)

    print(f"Visualizing {len(images)} images...")

    for img_info in images:
        img_id = img_info['id']
        filename = img_info['file_name']
        
        img_path = os.path.join(img_dir, filename)
        if not os.path.exists(img_path):
            img_path = os.path.join(img_dir, 'images', filename)
        
        full_img_bgr = cv2.imread(img_path)
        if full_img_bgr is None:
            print(f"Error loading {filename}")
            continue
            
        full_img_rgb = cv2.cvtColor(full_img_bgr, cv2.COLOR_BGR2RGB)
        h, w = full_img_rgb.shape[:2]
        
        anns = img_to_anns[img_id][:max_crops_per_img]
        cols = 1 + len(anns)
        
        _, axes = plt.subplots(1, cols, figsize=(15, 10))
        axes = np.atleast_1d(axes)

        main_view = full_img_rgb.copy()
        for ann in anns:
            mask = _decode_mask(ann['segmentation'], h, w)
            
            if mask is not None:
                contours, _ = cv2.findContours((mask * 255).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(main_view, contours, -1, (0, 255, 0), 2)
                x, y, bw, bh = map(int, ann['bbox'])
                cv2.rectangle(main_view, (x, y), (x+bw, y+bh), (255, 0, 0), 2)

        axes[0].imshow(main_view)
        axes[0].set_title(f"{filename}\n({len(anns)} objects)")
        axes[0].axis('off')

        for i, ann in enumerate(anns):
            mask = _decode_mask(ann['segmentation'], h, w)
            masked_img = full_img_rgb * mask[:, :, None]
            
            x, y, bw, bh = map(int, ann['bbox'])
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(w, x + bw), min(h, y + bh)
            
            if x2 > x1 and y2 > y1:
                crop = masked_img[y1:y2, x1:x2]
                axes[i+1].imshow(crop)
                axes[i+1].set_title(f"ID: {ann.get('id', i)}\nArea: {int(ann['area'])}")
            else:
                axes[i+1].text(0.5, 0.5, "Invalid Box", ha='center')
            axes[i+1].axis('off')

        plt.tight_layout()
        plt.show()

def _decode_mask(segmentation, h, w):
    if isinstance(segmentation, dict):
        return mask_util.decode(segmentation)
    elif isinstance(segmentation, list):
        mask = np.zeros((h, w), dtype=np.uint8)
        for seg in segmentation:
            poly = np.array(seg).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(mask, [poly], 1)
        return mask
    return None

plot_annotations_with_crops(OUTPUT_JSON, IMAGE_DIR, num_images=3)
